# NautilusTrader QuickStart Analysis

This notebook demonstrates interactive backtesting and analysis.

**What is Jupyter?** It's like a notebook where you can:
- Write code in cells
- Run each cell independently (Shift+Enter)
- See results immediately below each cell
- Mix code, charts, and notes

**Use Cases:**
- Exploring data interactively
- Testing strategy ideas quickly
- Creating visualizations
- Documenting research

## 1. Setup and Imports

Run this cell first to import all necessary libraries.

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
from decimal import Decimal

# NautilusTrader imports
from nautilus_trader.adapters.binance import BINANCE_VENUE
from nautilus_trader.backtest.config import BacktestEngineConfig
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import LoggingConfig
from nautilus_trader.examples.algorithms.twap import TWAPExecAlgorithm
from nautilus_trader.examples.strategies.ema_cross_twap import EMACrossTWAP, EMACrossTWAPConfig
from nautilus_trader.model.currencies import ETH, USDT
from nautilus_trader.model.data import BarType
from nautilus_trader.model.enums import AccountType, BookType, OmsType
from nautilus_trader.model.identifiers import TraderId
from nautilus_trader.model.objects import Money
from nautilus_trader.persistence.wranglers import TradeTickDataWrangler
from nautilus_trader.test_kit.providers import TestDataProvider, TestInstrumentProvider

# Set pandas display options for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

print("✅ Libraries imported successfully!")

## 2. Configure and Build Backtest Engine

Set up the backtesting environment.

In [ ]:
# Configure backtest engine
config = BacktestEngineConfig(
    trader_id=TraderId("BACKTESTER-001"),
    logging=LoggingConfig(log_level="ERROR", use_pyo3=False),
)

# Build engine
engine = BacktestEngine(config=config)

# Add venue
engine.add_venue(
    venue=BINANCE_VENUE,
    oms_type=OmsType.NETTING,
    book_type=BookType.L1_MBP,
    account_type=AccountType.CASH,
    base_currency=None,
    starting_balances=[Money(1_000_000.0, USDT), Money(10.0, ETH)],
    trade_execution=True,
)

print("✅ Backtest engine configured")
print(f"   Venue: {BINANCE_VENUE}")
print(f"   Starting Balance: 1,000,000 USDT + 10 ETH")

## 3. Load Data

Load historical market data for backtesting.

In [ ]:
# Add instrument
ETHUSDT_BINANCE = TestInstrumentProvider.ethusdt_binance()
engine.add_instrument(ETHUSDT_BINANCE)

# Load data
provider = TestDataProvider()
wrangler = TradeTickDataWrangler(instrument=ETHUSDT_BINANCE)
ticks = wrangler.process(provider.read_csv_ticks("binance/ethusdt-trades.csv"))
engine.add_data(ticks)

print(f"✅ Loaded {len(ticks):,} trade ticks")
print(f"   Instrument: {ETHUSDT_BINANCE.id}")
print(f"   First tick: {pd.Timestamp(ticks[0].ts_init, unit='ns')}")
print(f"   Last tick:  {pd.Timestamp(ticks[-1].ts_init, unit='ns')}")

## 4. Configure Strategy

Try modifying the parameters below and see how results change!

In [ ]:
# Strategy configuration - EXPERIMENT WITH THESE!
strategy_config = EMACrossTWAPConfig(
    instrument_id=ETHUSDT_BINANCE.id,
    bar_type=BarType.from_str("ETHUSDT.BINANCE-250-TICK-LAST-INTERNAL"),
    trade_size=Decimal("0.10"),      # Try: 0.05, 0.10, 0.20
    fast_ema_period=10,               # Try: 5, 10, 15
    slow_ema_period=20,               # Try: 20, 30, 50
    twap_horizon_secs=10.0,
    twap_interval_secs=2.5,
)

# Instantiate strategy and execution algorithm
strategy = EMACrossTWAP(config=strategy_config)
exec_algorithm = TWAPExecAlgorithm()

engine.add_strategy(strategy=strategy)
engine.add_exec_algorithm(exec_algorithm)

print("✅ Strategy configured")
print(f"   Fast EMA: {strategy_config.fast_ema_period}")
print(f"   Slow EMA: {strategy_config.slow_ema_period}")
print(f"   Position Size: {strategy_config.trade_size} ETH")

## 5. Run Backtest

Execute the backtest with our configured strategy.

In [ ]:
import time

start = time.time()
engine.run()
elapsed = time.time() - start

print(f"✅ Backtest complete in {elapsed:.2f} seconds")
print(f"   Processed {len(ticks):,} ticks")
print(f"   Speed: {len(ticks)/elapsed:,.0f} ticks/second")

## 6. Analyze Results - Positions Report

View all trading positions and their P&L.

In [ ]:
# Generate positions report
positions_df = engine.trader.generate_positions_report()

print(f"Total Positions: {len(positions_df)}")
print("\nPositions Summary:")
positions_df[['entry', 'avg_px_open', 'avg_px_close', 'realized_pnl', 'commissions']]

## 7. Performance Statistics

Calculate key performance metrics.

In [ ]:
# Get positions from cache
positions = engine.cache.positions()
closed_positions = [p for p in positions if p.is_closed]

# Calculate statistics
total_trades = len(closed_positions)
winners = [p for p in closed_positions if p.realized_pnl.as_double() > 0]
losers = [p for p in closed_positions if p.realized_pnl.as_double() < 0]

total_pnl = sum(p.realized_pnl.as_double() for p in closed_positions)
win_rate = (len(winners) / total_trades * 100) if total_trades > 0 else 0

avg_win = sum(p.realized_pnl.as_double() for p in winners) / len(winners) if winners else 0
avg_loss = sum(p.realized_pnl.as_double() for p in losers) / len(losers) if losers else 0

# Create summary dataframe
summary = pd.DataFrame({
    'Metric': [
        'Total Trades',
        'Winning Trades',
        'Losing Trades',
        'Win Rate %',
        'Total P&L (USDT)',
        'Avg Win (USDT)',
        'Avg Loss (USDT)',
        'Best Trade (USDT)',
        'Worst Trade (USDT)',
    ],
    'Value': [
        total_trades,
        len(winners),
        len(losers),
        f"{win_rate:.2f}%",
        f"{total_pnl:.8f}",
        f"{avg_win:.8f}",
        f"{avg_loss:.8f}",
        f"{max([p.realized_pnl.as_double() for p in closed_positions]):.8f}" if closed_positions else "N/A",
        f"{min([p.realized_pnl.as_double() for p in closed_positions]):.8f}" if closed_positions else "N/A",
    ]
})

print("\n📊 PERFORMANCE SUMMARY")
print("=" * 50)
summary

## 8. Visualize P&L Distribution

See the distribution of winning and losing trades.

In [ ]:
# Uncomment if you have matplotlib installed:
# pip install matplotlib

try:
    import matplotlib.pyplot as plt
    
    pnls = [p.realized_pnl.as_double() for p in closed_positions]
    
    plt.figure(figsize=(12, 5))
    
    # Histogram
    plt.subplot(1, 2, 1)
    plt.hist(pnls, bins=15, edgecolor='black', alpha=0.7)
    plt.axvline(x=0, color='r', linestyle='--', linewidth=2, label='Break-even')
    plt.xlabel('P&L (USDT)')
    plt.ylabel('Frequency')
    plt.title('P&L Distribution')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Cumulative P&L
    plt.subplot(1, 2, 2)
    cumulative_pnl = np.cumsum(pnls)
    plt.plot(cumulative_pnl, linewidth=2)
    plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    plt.xlabel('Trade Number')
    plt.ylabel('Cumulative P&L (USDT)')
    plt.title('Cumulative P&L')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("⚠️  matplotlib not installed. Run: pip install matplotlib")
    print("   Then re-run this cell to see visualizations")

## 9. Account Balance Over Time

Track how your account balance changed throughout the backtest.

In [ ]:
# Get account report
account_df = engine.trader.generate_account_report(BINANCE_VENUE)

# Filter for USDT balance changes
usdt_balance = account_df[account_df['currency'] == 'USDT'][['total']].copy()
usdt_balance.index = pd.to_datetime(usdt_balance.index)

print(f"Starting Balance: {usdt_balance.iloc[0]['total']:,.2f} USDT")
print(f"Ending Balance:   {usdt_balance.iloc[-1]['total']:,.2f} USDT")
print(f"Change:           {usdt_balance.iloc[-1]['total'] - usdt_balance.iloc[0]['total']:,.2f} USDT")

# Plot if matplotlib available
try:
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(14, 5))
    plt.plot(usdt_balance.index, usdt_balance['total'], linewidth=2)
    plt.axhline(y=1_000_000, color='r', linestyle='--', alpha=0.5, label='Starting Balance')
    plt.xlabel('Time')
    plt.ylabel('Balance (USDT)')
    plt.title('Account Balance Over Time')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    pass

## 10. Experiment: Change Parameters

Now go back to **Cell 4** and try different parameters:

**Ideas to test:**
1. Change `fast_ema_period` from 10 to 5
2. Change `slow_ema_period` from 20 to 50
3. Change `trade_size` from 0.10 to 0.05

Then:
- Re-run cells 4 through 9
- Compare the results
- Which parameters performed better?

**This is the power of Jupyter** - quick experimentation and instant feedback!

## 11. Clean Up

Reset the engine for the next run.

In [ ]:
engine.reset()
engine.dispose()
print("✅ Engine reset and disposed")

---

## 🎓 What You Learned

✅ How to use Jupyter notebooks interactively  
✅ Running backtests step-by-step  
✅ Analyzing performance metrics  
✅ Visualizing results with charts  
✅ Experimenting with parameters  

## 📖 Next Steps

1. **Read** `LEARNING_PATH.md` for structured learning
2. **Study** example strategies in `/examples/strategies/`
3. **Practice** running `/examples/backtest/example_01/` through `example_11/`
4. **Build** your own strategy!

---

**Happy Trading! 🚀**